In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 87 — Procurement Negotiation Copilot

## Goal
Analyze **vendor proposals**, compare them against benchmarks,
and **suggest negotiation levers** — pricing, terms, SLAs.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Proposal analysis** | Extract and compare vendor terms |
| **Benchmarking** | Compare against market rates |
| **Negotiation strategy** | AI-suggested levers |
| **CrewAI** | Multi-agent procurement team |

## Stack
- `crewai` — multi-agent orchestration
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from crewai import Agent, Task, Crew
from crewai.tools import tool

print("All imports OK")

## Step 1 — Vendor Proposals & Market Benchmarks

In [ ]:
# Two vendor proposals for comparison
PROPOSALS = {
    "Vendor A (CloudStack Pro)": {
        "total_cost": "$480,000/year (3-year contract)",
        "per_user_cost": "$40/user/month (1,000 users)",
        "setup_fee": "$50,000 one-time",
        "sla_uptime": "99.9%",
        "sla_response": "4 hours for P1, 8 hours for P2",
        "data_location": "US only",
        "contract_term": "3 years, auto-renew",
        "termination": "180-day notice, 50% of remaining contract as penalty",
        "price_escalation": "5% annual increase",
        "included_support": "Standard (email + chat, business hours)",
        "premium_support": "$60,000/year additional for 24/7",
        "migration_support": "Included (up to 500 hours)",
        "certifications": ["SOC 2 Type II", "ISO 27001", "GDPR"]
    },
    "Vendor B (DataNimbus)": {
        "total_cost": "$420,000/year (2-year contract)",
        "per_user_cost": "$35/user/month (1,000 users)",
        "setup_fee": "$30,000 one-time",
        "sla_uptime": "99.95%",
        "sla_response": "2 hours for P1, 4 hours for P2",
        "data_location": "US and EU",
        "contract_term": "2 years, no auto-renew",
        "termination": "90-day notice, no penalty after year 1",
        "price_escalation": "3% annual increase (capped)",
        "included_support": "Premium (24/7 phone + chat)",
        "premium_support": "Included",
        "migration_support": "$25,000 additional",
        "certifications": ["SOC 2 Type II", "HIPAA", "FedRAMP Moderate"]
    }
}

# Market benchmarks
BENCHMARKS = {
    "per_user_cost_range": "$25-$45/user/month for enterprise SaaS",
    "typical_sla": "99.95% is premium tier, 99.9% is standard",
    "setup_fee_range": "$20K-$75K depending on complexity",
    "typical_escalation": "3-5% annual, best-in-class is CPI-indexed",
    "termination_best_practice": "90-day notice, no penalty after first year",
    "support_benchmark": "24/7 support typically costs $40K-$80K/year as add-on"
}

print(f"Loaded {len(PROPOSALS)} vendor proposals")

In [ ]:
# Tools for the agents
@tool
def get_vendor_proposals(query: str) -> str:
    """Get all vendor proposal details for comparison."""
    return json.dumps(PROPOSALS, indent=2)

@tool
def get_market_benchmarks(query: str) -> str:
    """Get market benchmark data for enterprise SaaS pricing and terms."""
    return json.dumps(BENCHMARKS, indent=2)

print("Tools defined")

## Step 2 — Procurement Agents

In [ ]:
proposal_analyst = Agent(
    role="Proposal Analyst",
    goal="Compare vendor proposals side-by-side on price, terms, and capabilities",
    backstory="""You are an experienced procurement analyst who has evaluated
    hundreds of enterprise software deals. You excel at finding hidden costs,
    unfavorable terms, and comparison gaps between vendors.""",
    tools=[get_vendor_proposals],
    llm="ollama/qwen3.5:9b",
    verbose=True,
    max_iter=3
)

benchmark_analyst = Agent(
    role="Market Benchmark Analyst",
    goal="Compare proposals against market benchmarks to identify negotiation opportunities",
    backstory="""You specialize in enterprise SaaS market intelligence.
    You know what top companies pay and what terms are standard vs premium.
    You spot where vendors are above or below market.""",
    tools=[get_market_benchmarks, get_vendor_proposals],
    llm="ollama/qwen3.5:9b",
    verbose=True,
    max_iter=3
)

negotiation_strategist = Agent(
    role="Negotiation Strategist",
    goal="Develop a negotiation strategy with specific levers and tactics",
    backstory="""You are a former VP of Procurement who has negotiated
    $500M+ in enterprise contracts. You know exactly which levers to pull,
    how to create competitive tension, and when to walk away.""",
    llm="ollama/qwen3.5:9b",
    verbose=True,
    max_iter=3
)

print("3 procurement agents defined")

In [ ]:
comparison_task = Task(
    description="""Use the get_vendor_proposals tool to retrieve both proposals.
    Create a detailed side-by-side comparison covering:
    1. Total cost of ownership (3-year view)
    2. SLA comparison
    3. Contract flexibility
    4. Support and migration
    5. Compliance certifications
    Identify the strengths and weaknesses of each vendor.""",
    expected_output="A structured comparison table with pros/cons for each vendor",
    agent=proposal_analyst
)

benchmark_task = Task(
    description="""Use the market benchmarks tool and vendor proposals tool.
    For each proposal term, assess whether it's:
    - Below market (favorable)
    - At market (standard)
    - Above market (unfavorable / negotiable)
    Quantify potential savings where possible.""",
    expected_output="A benchmark analysis with savings opportunities quantified",
    agent=benchmark_analyst
)

strategy_task = Task(
    description="""Based on the comparison and benchmark analysis, create a
    negotiation strategy that includes:
    1. Recommended vendor (with reasoning)
    2. Top 5 negotiation levers ranked by impact
    3. Specific counter-offer terms
    4. Walk-away points
    5. Competitive tension tactics
    6. Expected savings from negotiation""",
    expected_output="A complete negotiation playbook with specific asks and tactics",
    agent=negotiation_strategist,
    context=[comparison_task, benchmark_task]
)

print("Tasks defined")

In [ ]:
# Run the procurement crew
crew = Crew(
    agents=[proposal_analyst, benchmark_analyst, negotiation_strategist],
    tasks=[comparison_task, benchmark_task, strategy_task],
    verbose=True
)

print("Running procurement analysis...\n")
result = crew.kickoff()

print("\n" + "=" * 70)
print("NEGOTIATION STRATEGY")
print("=" * 70)
print(result)

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Proposal parsing** | Extract and compare structured terms |
| **Benchmarking** | Market rate comparison per term |
| **Strategy generation** | AI-powered negotiation playbook |
| **CrewAI** | 3 specialized procurement agents |

## Architecture
```
Vendor Proposals + Benchmarks
          ↓
Proposal Analyst → Side-by-Side Comparison
          ↓
Benchmark Analyst → Market Position Analysis
          ↓
Negotiation Strategist → Playbook (levers, counter-offers, walk-away)
```

## Extending This Project
- Add real-time market pricing APIs
- Score proposals with weighted criteria
- Generate counter-proposal documents
- Track negotiation rounds and concessions